In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_2016_2022_bn.csv")

In [3]:
df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_lt

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,EC,DI,SSB,COC,CPI,EUPC
0,2.74,4.9,4.4,7.98,46.0,-1.27,26.00,20.287640,17.029185,3.877441,...,74.3,0.01,312.00,25.0,78.802445,7.47,92.931332,0.68,59.0,1.155739e+08
1,2.90,4.7,4.4,7.78,45.0,-1.40,25.72,20.480116,16.299364,5.888726,...,75.5,-0.01,294.85,-79.0,78.802445,7.41,92.984744,0.52,59.0,1.529936e+08
2,2.40,4.7,4.6,7.18,45.0,-0.95,25.58,22.406639,16.217817,5.790665,...,76.7,0.00,-161.65,86.0,78.802445,7.50,93.053500,0.47,59.0,2.870759e+08
3,6.23,4.8,4.9,6.44,45.0,-0.26,25.26,19.580386,16.795948,4.979384,...,77.4,0.01,350.20,128.0,78.802445,7.50,93.175686,0.67,60.0,1.883568e+08
4,7.94,4.7,4.4,5.69,45.0,0.03,24.79,16.519823,18.360545,0.068446,...,75.8,0.01,150.30,5.0,78.802445,7.13,93.273709,0.78,60.0,5.145216e+08
5,3.74,4.7,4.4,5.31,45.0,0.21,24.86,21.835007,17.497495,6.455358,...,76.7,0.00,-35.80,33.0,78.802445,7.18,93.083794,0.82,61.0,4.056306e+08
6,3.41,4.7,4.4,5.46,45.0,1.09,25.64,28.075306,16.744046,1.695286,...,78.6,0.02,151.65,33.0,78.802445,7.31,93.083794,0.75,62.0,3.312880e+08


In [5]:
discretised_lt = pd.DataFrame(index=df_lt.index)

for col in df_lt.columns:
    no_unique = df_lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                df_lt[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = df_lt[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    0    0     2     0    0     2     0     1      0  ...    0    1   
1    0    0    0     2     0    0     2     1     0      2  ...    0    0   
2    0    0    0     1     0    0     1     2     0      1  ...    1    0   
3    2    0    0     1     0    1     0     0     1      1  ...    2    1   
4    2    0    0     0     0    1     0     0     2      0  ...    0    1   
5    1    0    0     0     0    2     0     1     2      2  ...    1    0   
6    1    0    0     0     0    2     1     2     0      0  ...    2    2   

   EPFRG  GGAG  EC  DI  SSB  COC  CPI  EUPC  
0      2     0   0   1    0    1    0     0  
1      1     0   0   1    0    0    0     0  
2      0     2   0   2    0    0    0     1  
3      2     2   0   2    2    0    0     0  
4      0     0   0   0    2    2    0     2  
5      0     1   0   0    1    2    1     2  
6      1     1   0   0    1    1    1     1

In [6]:
sm = from_pandas(discretised_lt)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_lt.html")

Graphs/01_fully_connected_lt.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_lt.html")

Graphs/01_thresholded_lt.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_lt_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
21    SSB   EBSG  2.567989
26    CPI   EIAP  2.008391
27    CPI    PGP  2.000132
10    EIS    LFG  1.982487
34   EUPC   VABI  1.924496
6    EBSG  GDPCG  1.774248
13   LBGD   EIAP  1.770139
17  EPFRG   VABI  1.760424
7    GDPC    FDI  1.685740
4    GGFC  EPFRG  1.654928
18   GGAG    EII  1.499689
29    CPI    EII  1.495684
28    CPI  GDPCG  1.450312
5     EII   EUPC  1.349547
11    EIS  EPFRG  1.205119
31    CPI    LTU  1.131250
16    FEM    EMP  0.999994
12    EIS   EUPC  0.999801
24    COC   GGFC  0.999594
2     PGP   GDPC  0.996508
32    CPI    FEM  0.965167
15    EMP   GGAG  0.954860
1    EIAP    LTU  0.859574
35   EUPC   EBSG  0.856738
0     FDI    SSB  0.842248
30    CPI    EIS  0.838682
19    SSB   GGFC  0.829279
23    SSB  EPFRG -0.821288
14    EMP    EII -0.831762
22    SSB    LFG -0.982265
8     EIS   GGFC -1.000441
20    SSB  GDPCG -1.177486
3    GGFC   VABI -1.300902
33    CPI     DI -1.592558
36   EUPC  EPFRG -2.012675
9     EIS   EBSG -2.066353
2

In [11]:
df_edges.to_csv("lt_2016_2022.csv")